# 07 — Warehouse Analysis

Example queries for Sayli & Shraddha — **no API keys needed.**

This notebook reads from `data/warehouse/solar.duckdb`, which is populated by `python3 solar_etl.py --location ...`. Just clone the repo, pull the latest `.duckdb` file (or ask Scott to run some ETL), and query.

See [docs/WAREHOUSE.md](../docs/WAREHOUSE.md) for the full schema.


## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from solar_warehouse import get_conn, table_counts
import pandas as pd

conn = get_conn('../data/warehouse/solar.duckdb')

# Row counts for all tables
for name, count in sorted(table_counts(conn).items()):
    print(f'  {name:20s}  {count:>6} rows')

## 1. Raw quote results

In [ ]:
# Every quote we've run, with top-line metrics
df = conn.execute('''
    SELECT
        location_query, state, zip_code,
        round(viability_score, 1) as score,
        round(payback_years, 1) as payback,
        round(electricity_rate_used, 3) as rate,
        rate_source,
        confidence_level
    FROM raw_quote
    ORDER BY viability_score DESC
''').df()
df

## 2. Star-schema join (fact_quote ← dim_*)

In [ ]:
# Quotes joined to their dimensions — the classic OLAP query
df = conn.execute('''
    SELECT
        l.zip_code,
        l.state,
        u.utility_name,
        t.rate_name,
        round(t.flat_rate, 3) as tariff_rate,
        round(f.viability_score, 1) as score,
        round(f.npv_25yr, 0) as npv,
        round(f.payback_years, 1) as payback
    FROM fact_quote f
    LEFT JOIN dim_location l ON l.location_id = f.location_id
    LEFT JOIN dim_utility u ON u.utility_id = f.utility_id
    LEFT JOIN dim_tariff t ON t.tariff_id = f.tariff_id
    ORDER BY f.viability_score DESC
''').df()
df

## 3. Viability score distribution by state

In [ ]:
import matplotlib.pyplot as plt

df = conn.execute('''
    SELECT state, viability_score
    FROM raw_quote
    WHERE state IS NOT NULL
''').df()

if len(df) > 0:
    df.boxplot(column='viability_score', by='state')
    plt.title('Viability Score by State')
    plt.suptitle('')
    plt.ylabel('Score (0-100)')
    plt.show()
else:
    print('No quotes yet. Run `python3 solar_etl.py --location ...` first.')

## 4. Rate source breakdown — where are our numbers coming from?

In [ ]:
conn.execute('''
    SELECT
        rate_source,
        count(*) as n_quotes,
        round(avg(electricity_rate_used), 3) as avg_rate,
        round(avg(viability_score), 1) as avg_score
    FROM raw_quote
    GROUP BY rate_source
    ORDER BY n_quotes DESC
''').df()

## 5. SCD Type 2 — tariff history over time

`dim_tariff` tracks each (utility, rate_name) with `effective_from` / `effective_to`. A NULL `effective_to` means it's currently in effect.

In [ ]:
conn.execute('''
    SELECT
        u.utility_name,
        t.rate_name,
        t.flat_rate,
        t.effective_from,
        t.effective_to,
        CASE WHEN t.effective_to IS NULL THEN 'current' ELSE 'historical' END as status
    FROM dim_tariff t
    LEFT JOIN dim_utility u ON u.utility_id = t.utility_id
    ORDER BY u.utility_name, t.effective_from
''').df()

## Cleanup

In [ ]:
conn.close()